In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision.ops import roi_align
import torchvision.transforms as T
from datasets import Dataset, DatasetDict, Image as HFImage
from PIL import Image
import os
import gzip
from tqdm import tqdm

# --- 1. Setup and Constants ---
print("--- Setting up environment for V2 Object Feature Extraction ---")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# This is the directory where we will save the new features
OBJ_FEATURE_DIR = "object_features"

# We will extract the top 20 objects per image
K_TOP_OBJECTS = 20
# We will only keep objects with a confidence score > 0.5
CONF_THRESHOLD = 0.5

--- Setting up environment for V2 Object Feature Extraction ---
Using device: cuda


In [2]:
# --- 2. Load Models ---
print("Loading pre-trained models...")

# Model 1: The Detector
# We use a standard Faster R-CNN to get bounding boxes.
model_detector = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model_detector = model_detector.to(DEVICE).eval()
print("Detector (Faster R-CNN) loaded.")

# Model 2: The Feature Extractor
# We use the same ResNet-50 from your 2.ipynb, but we stop
# *before* the final avgpool and fc layers.
# We want the rich 2D feature map, not a flat 2048-dim vector.
model_resnet = torchvision.models.resnet50(pretrained=True)
model_extractor = nn.Sequential(*list(model_resnet.children())[:-2])
model_extractor = model_extractor.to(DEVICE).eval()
print("Extractor (ResNet-50 backbone) loaded.")

Loading pre-trained models...


c:\Users\Hp\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Hp\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Hp/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:14<00:00, 11.6MB/s] 


Detector (Faster R-CNN) loaded.
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\Hp/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


c:\Users\Hp\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 97.8M/97.8M [00:08<00:00, 11.7MB/s]


Extractor (ResNet-50 backbone) loaded.


In [3]:
# --- 3. Define Image Transforms ---

# Transform for the detector (just convert to tensor)
T_detector = T.ToTensor()

# Transform for the feature extractor (normalize like ImageNet)
T_extractor = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225])
])
print("Image transforms defined.")

Image transforms defined.


In [4]:
# --- 4. Load Dataset (for image paths) ---
from datasets import Dataset, DatasetDict, Image as HFImage
import gzip

print("--- Re-creating the mapped dataset from local files ---")
image_folder_path = os.path.join("flickr30k", "Images")
repo_path = "dataset"
final_dataset = None

dataset_dict = {}
split_map = {
    "train": {"en": "train.en.gz", "de": "train.de.gz", "files": "train.txt"},
    "validation": {"en": "val.en.gz", "de": "val.de.gz", "files": "val.txt"},
    "test": {"en": "test_2016_flickr.en.gz", "de": "test_2016_flickr.de.gz", "files": "test_2016_flickr.txt"}
}
text_path = os.path.join(repo_path, "data", "task1", "raw")
files_path = os.path.join(repo_path, "data", "task1", "image_splits")

for split_name, paths in split_map.items():
    with gzip.open(os.path.join(text_path, paths['en']), 'rt', encoding='utf-8') as f: en_sents = f.read().strip().split('\n')
    with gzip.open(os.path.join(text_path, paths['de']), 'rt', encoding='utf-8') as f: de_sents = f.read().strip().split('\n')
    with open(os.path.join(files_path, paths['files']), 'r', encoding='utf-8') as f: img_files = f.read().strip().split('\n')
    data = {'image': [os.path.join(image_folder_path, fname) for fname in img_files], 'en': en_sents, 'de': de_sents}
    dataset_dict[split_name] = Dataset.from_dict(data).cast_column("image", HFImage())

final_dataset = DatasetDict(dataset_dict)
print("✅ Dataset object created successfully.")

--- Re-creating the mapped dataset from local files ---
✅ Dataset object created successfully.


In [6]:
# --- 5. Main Extraction Loop ---
print(f"Starting feature extraction. This will take a while...")
print(f"Output will be saved to '{OBJ_FEATURE_DIR}'")

# Ensure we are not calculating gradients, which saves memory
with torch.no_grad():
    for split in ['train', 'validation', 'test']:
        
        # Create the output directories (e.g., 'object_features/train')
        output_dir = os.path.join(OBJ_FEATURE_DIR, split)
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\nProcessing '{split}' split...")
        dataset_split = final_dataset[split]
        
        for idx, example in enumerate(tqdm(dataset_split)):
            
            # Define the path to save the new feature
            output_path = os.path.join(output_dir, f"{idx}.pt")
            
            # Simple check to skip files we've already processed
            if os.path.exists(output_path):
                continue
                
            # === CHANGE ===
            # The 'try' block now wraps the image access itself,
            # as this is where the FileNotFoundError can happen.
            try:
                # 1. Load the single image
                # 'example['image']' directly returns a PIL.Image object
                # because we used .cast_column("image", HFImage())
                image_object = example['image']
                
                # Handle cases where the image file was corrupt or unreadable
                if image_object is None:
                    print(f"Warning: Could not load image at index {idx}. Skipping.")
                    continue
                    
                # We just need to convert it, not open it again
                image = image_object.convert("RGB")
                
            except FileNotFoundError:
                # We can try to get the path from the object for a better error message
                image_path_str = "Unknown"
                if hasattr(example['image'], 'path'):
                     image_path_str = example['image'].path
                print(f"Warning: Could not find image file at {image_path_str}. Skipping.")
                continue
            # === END CHANGE ===

            # 2. Prepare images for both models
            image_tensor_detector = T_detector(image).to(DEVICE)
            image_tensor_extractor = T_extractor(image).unsqueeze(0).to(DEVICE) # Add batch dim

            # 3. Get Bounding Boxes from Detector
            detector_output = model_detector([image_tensor_detector])
            boxes = detector_output[0]['boxes']
            scores = detector_output[0]['scores']

            # 4. Filter boxes by confidence and take the top K
            keep = (scores > CONF_THRESHOLD)
            final_boxes = boxes[keep][:K_TOP_OBJECTS]

            # 5. Get the 2D Feature Map from the Extractor
            #    Shape will be [1, 2048, H/32, W/32]
            feature_map = model_extractor(image_tensor_extractor)

            # 6. Handle the (rare) case of no objects found
            if final_boxes.shape[0] == 0:
                # We save a single "background" feature
                object_features = torch.zeros(1, 2048, device=DEVICE)
            else:
                # 7. --- This is the key step ---
                # Use RoI Align to extract features for each box
                # We must tell roi_align how much the feature map was downscaled
                # (ResNet-50 downscales by 32)
                object_feature_patches = roi_align(
                    input=feature_map,
                    boxes=[final_boxes],
                    output_size=(1, 1), # Pool each RoI to a 1x1 feature
                    spatial_scale=(1.0 / 32.0)
                )
                # Shape is now [NumBoxes, 2048, 1, 1]
                
                # 8. Squeeze to final shape: [NumBoxes, 2048]
                object_features = object_feature_patches.squeeze(-1).squeeze(-1)

            # 9. Save the final tensor to disk
            torch.save(object_features.cpu(), output_path)

print("\n\n✅ All features extracted successfully!")

Starting feature extraction. This will take a while...
Output will be saved to 'object_features'

Processing 'train' split...


100%|██████████| 29000/29000 [53:44<00:00,  8.99it/s]



Processing 'validation' split...


100%|██████████| 1014/1014 [01:53<00:00,  8.97it/s]



Processing 'test' split...


100%|██████████| 1000/1000 [02:04<00:00,  8.05it/s]




✅ All features extracted successfully!
